# 🗺️ Day 03: 2D Dynamic Programming — Grids & Two Dimensions

---

## When Does DP Become 2D?

When your state needs **two parameters** to describe it:
- Grid problems: `dp[row][col]`
- Two-sequence problems: `dp[i][j]` for positions in string1 and string2
- Capacity problems: `dp[item][capacity]`

**The mental model:** You're filling a 2D table. Each cell `dp[i][j]` is computed from cells above, to the left, or diagonally.

```
        j→
   i  ┌───┬───┬───┬───┐
   ↓  │ B │ ← │ ← │ ← │   Cell uses values from
      ├───┼───┼───┼───┤   ↑ (above), ← (left), or ↖ (diagonal)
      │ ↑ │ ↖ │   │   │
      ├───┼───┼───┼───┤
      │ ↑ │   │   │   │
      └───┴───┴───┴───┘
```

---

## 🎯 Problem 1: Unique Paths (LC #62)

Robot at top-left of m×n grid. Can only move right or down. How many unique paths to bottom-right?

**State:** `dp[i][j]` = number of paths to reach cell `(i, j)`.

**Transition:** You came from above or from the left:
```
dp[i][j] = dp[i-1][j] + dp[i][j-1]
```

**Base:** First row and first column are all 1 (only one way to reach them).

In [ ]:
def uniquePaths(m, n):
    """
    Space-optimized: only need previous row.
    Time: O(m*n), Space: O(n)
    """
    dp = [1] * n  # First row: all 1s
    
    for i in range(1, m):
        for j in range(1, n):
            dp[j] += dp[j-1]  # dp[j] (above) + dp[j-1] (left)
    
    return dp[-1]

print(uniquePaths(3, 7))  # 28
print(uniquePaths(3, 2))  # 3

**Space trick:** Since each row only depends on the current and previous row, we can reuse a single 1D array. `dp[j]` before update = "above", `dp[j-1]` after update = "left".

---

## 🎯 Problem 2: Minimum Path Sum (LC #64)

Same grid, but each cell has a cost. Find minimum cost path from top-left to bottom-right.

**Transition:** `dp[i][j] = grid[i][j] + min(dp[i-1][j], dp[i][j-1])`

In [ ]:
def minPathSum(grid):
    """
    In-place DP on the grid itself (or use 1D dp array).
    Time: O(m*n), Space: O(1) if modifying grid
    """
    m, n = len(grid), len(grid[0])
    
    # Fill first row (can only come from left)
    for j in range(1, n):
        grid[0][j] += grid[0][j-1]
    # Fill first column (can only come from above)
    for i in range(1, m):
        grid[i][0] += grid[i-1][0]
    # Fill rest
    for i in range(1, m):
        for j in range(1, n):
            grid[i][j] += min(grid[i-1][j], grid[i][j-1])
    
    return grid[-1][-1]

print(minPathSum([[1,3,1],[1,5,1],[4,2,1]]))  # 7 (1→3→1→1→1)

---

## 🎯 Problem 3: Maximal Square (LC #221)

Find the largest square of 1s in a binary matrix. Return its area.

**State:** `dp[i][j]` = side length of largest square with bottom-right corner at `(i, j)`.

**Key insight:** A square of side `k` at `(i,j)` requires squares of side `k-1` at three neighbors.

```
dp[i][j] = min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1]) + 1
```

Why min? The square is limited by its weakest link — the smallest neighboring square.

In [ ]:
def maximalSquare(matrix):
    """
    2D DP: each cell = size of largest square ending here.
    Time: O(m*n), Space: O(n)
    """
    if not matrix:
        return 0
    m, n = len(matrix), len(matrix[0])
    dp = [0] * (n + 1)  # 1-indexed for cleaner boundary
    max_side = 0
    prev = 0  # dp[i-1][j-1]
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            temp = dp[j]  # Save before overwrite (this will be prev for next j)
            if matrix[i-1][j-1] == '1':
                dp[j] = min(dp[j], dp[j-1], prev) + 1
                max_side = max(max_side, dp[j])
            else:
                dp[j] = 0
            prev = temp
        prev = 0  # Reset for new row
    
    return max_side * max_side

matrix = [
    ['1','0','1','0','0'],
    ['1','0','1','1','1'],
    ['1','1','1','1','1'],
    ['1','0','0','1','0']
]
print(maximalSquare(matrix))  # 4 (2x2 square)

---

## 🗺️ 2D DP Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  2D DP PATTERNS                                                     ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  GRID PATHFINDING:                                                  ║
║    dp[i][j] = f(dp[i-1][j], dp[i][j-1])                           ║
║    → Unique Paths, Min Path Sum                                    ║
║                                                                      ║
║  SQUARE / RECTANGLE:                                                ║
║    dp[i][j] = min(top, left, diagonal) + 1                         ║
║    → Maximal Square, Maximal Rectangle (uses histogram trick)      ║
║                                                                      ║
║  SPACE OPTIMIZATION: 2D → 1D if only previous row needed          ║
║    Keep dp[n] array, update left-to-right for each row             ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 62 | Unique Paths | Medium | Count paths: sum(above, left) |
| 63 | Unique Paths II | Medium | Same + obstacles (set to 0) |
| 64 | Minimum Path Sum | Medium | Min cost: grid[i][j] + min(above, left) |
| 221 | Maximal Square | Medium | min(3 neighbors) + 1 |
| 174 | Dungeon Game | Hard | DP from bottom-right (reverse direction!) |
| 85 | Maximal Rectangle | Hard | Histogram per row + stack |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 2D DP: state needs 2 parameters (row/col, index1/index2)      │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Grid DP: fill first row + first col, then rest cell by cell   │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Space: 2D → 1D array when only previous row needed           │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Maximal Square: min of 3 neighbors + 1 (classic pattern)      │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: String DP

LCS, Edit Distance, and palindrome problems — the bread and butter of string DP.